In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Enhanced Prenatal Risk Classifier\n",
    "\n",
    "This notebook demonstrates the complete implementation of a fetal health classification model with:\n",
    "- Feature engineering\n",
    "- Hyperparameter optimization\n",
    "- Comprehensive evaluation capabilities"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Import Required Libraries"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.impute import SimpleImputer\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.model_selection import GridSearchCV\n",
    "from sklearn.metrics import precision_score, recall_score, f1_score"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## PrenatalRiskClassifier Class Definition"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class PrenatalRiskClassifier:\n",
    "    def __init__(self):\n",
    "        \"\"\"Initialize fetal health classification model with feature engineering, \n",
    "        hyperparameter optimization, and comprehensive evaluation capabilities.\"\"\"\n",
    "        self.target_name = \"fetal_health\"\n",
    "\n",
    "        # Explicitly define expected features for robustness\n",
    "        self.features = [\n",
    "            'baseline value',\n",
    "            'accelerations',\n",
    "            'fetal_movement',\n",
    "            'uterine_contractions',\n",
    "            'light_decelerations',\n",
    "            'severe_decelerations',\n",
    "            'prolongued_decelerations',\n",
    "            'abnormal_short_term_variability',\n",
    "            'mean_value_of_short_term_variability',\n",
    "            'percentage_of_time_with_abnormal_long_term_variability',\n",
    "            'mean_value_of_long_term_variability',\n",
    "            'histogram_width',\n",
    "            'histogram_min',\n",
    "            'histogram_max',\n",
    "            'histogram_number_of_peaks',\n",
    "            'histogram_number_of_zeroes',\n",
    "            'histogram_mode',\n",
    "            'histogram_mean',\n",
    "            'histogram_median',\n",
    "            'histogram_variance',\n",
    "            'histogram_tendency'\n",
    "        ]\n",
    "        \n",
    "        # Store best parameters and grid search results\n",
    "        self.best_params_ = None\n",
    "        self.best_score_ = None\n",
    "        self.cv_results_ = None\n",
    "        self.best_estimator_ = None\n",
    "\n",
    "    def _engineer_features(self, X: pd.DataFrame) -> pd.DataFrame:\n",
    "        \"\"\"Create engineered features from existing features.\n",
    "        Returns DataFrame with original + engineered features.\"\"\"\n",
    "        X_eng = X.copy()\n",
    "        \n",
    "        # Feature 1: Deceleration ratio (total decelerations normalized)\n",
    "        deceleration_cols = ['light_decelerations', 'severe_decelerations', 'prolongued_decelerations']\n",
    "        if all(col in X_eng.columns for col in deceleration_cols):\n",
    "            X_eng['total_decelerations'] = (\n",
    "                X_eng['light_decelerations'] + \n",
    "                X_eng['severe_decelerations'] + \n",
    "                X_eng['prolongued_decelerations']\n",
    "            )\n",
    "            baseline_safe = X_eng['baseline value'].replace(0, np.nan)\n",
    "            X_eng['deceleration_baseline_ratio'] = X_eng['total_decelerations'] / (baseline_safe + 1e-6)\n",
    "        else:\n",
    "            X_eng['total_decelerations'] = 0.0\n",
    "            X_eng['deceleration_baseline_ratio'] = 0.0\n",
    "        \n",
    "        # Feature 2: Variability ratio (short-term vs long-term variability)\n",
    "        if 'mean_value_of_short_term_variability' in X_eng.columns and \\\n",
    "           'mean_value_of_long_term_variability' in X_eng.columns:\n",
    "            long_term_safe = X_eng['mean_value_of_long_term_variability'].replace(0, np.nan)\n",
    "            X_eng['variability_ratio'] = (\n",
    "                X_eng['mean_value_of_short_term_variability'] / (long_term_safe + 1e-6)\n",
    "            )\n",
    "        else:\n",
    "            X_eng['variability_ratio'] = 0.0\n",
    "        \n",
    "        # Feature 3: Histogram spread (range normalized by mean)\n",
    "        if all(col in X_eng.columns for col in ['histogram_min', 'histogram_max', 'histogram_mean']):\n",
    "            histogram_range = X_eng['histogram_max'] - X_eng['histogram_min']\n",
    "            mean_safe = X_eng['histogram_mean'].replace(0, np.nan)\n",
    "            X_eng['histogram_spread_ratio'] = histogram_range / (mean_safe + 1e-6)\n",
    "        else:\n",
    "            X_eng['histogram_spread_ratio'] = 0.0\n",
    "        \n",
    "        return X_eng\n",
    "\n",
    "    def _prepare_X(self, X: pd.DataFrame) -> pd.DataFrame:\n",
    "        \"\"\"Select expected features, ignore extra columns, create missing columns, \n",
    "        and apply feature engineering.\"\"\"\n",
    "        Xc = X.copy()\n",
    "\n",
    "        # Convert string columns to numeric where possible (robustness)\n",
    "        for col in Xc.columns:\n",
    "            if Xc[col].dtype == 'object':\n",
    "                try:\n",
    "                    Xc[col] = pd.to_numeric(Xc[col], errors='coerce')\n",
    "                except:\n",
    "                    pass\n",
    "\n",
    "        # Ignore extra / garbage columns safely\n",
    "        keep_cols = [c for c in self.features if c in Xc.columns]\n",
    "        Xc = Xc[keep_cols].copy()\n",
    "\n",
    "        # Ensure all expected feature columns exist\n",
    "        for c in self.features:\n",
    "            if c not in Xc.columns:\n",
    "                Xc[c] = np.nan\n",
    "\n",
    "        # Preserve the correct column order\n",
    "        Xc = Xc[self.features]\n",
    "        \n",
    "        # Apply feature engineering\n",
    "        Xc = self._engineer_features(Xc)\n",
    "\n",
    "        return Xc\n",
    "\n",
    "    def fit(self, X, y):\n",
    "        \"\"\"Fit the model with hyperparameter optimization using cross-validation.\"\"\"\n",
    "        Xp = self._prepare_X(X)\n",
    "        y_series = pd.Series(y).squeeze()\n",
    "        \n",
    "        # Create base pipeline\n",
    "        base_pipeline = Pipeline(steps=[\n",
    "            (\"imputer\", SimpleImputer(strategy=\"median\")),\n",
    "            (\"scaler\", StandardScaler()),\n",
    "            (\"model\", RandomForestClassifier(random_state=42, class_weight=\"balanced\"))\n",
    "        ])\n",
    "        \n",
    "        # Define hyperparameter grid\n",
    "        param_grid = {\n",
    "            'model__n_estimators': [200, 300, 400],\n",
    "            'model__max_depth': [10, 15, 20, None],\n",
    "            'model__min_samples_split': [2, 5, 10],\n",
    "            'model__min_samples_leaf': [1, 2, 4]\n",
    "        }\n",
    "        \n",
    "        # Perform grid search with 3-fold cross-validation\n",
    "        grid_search = GridSearchCV(\n",
    "            base_pipeline,\n",
    "            param_grid,\n",
    "            cv=3,\n",
    "            scoring='f1_macro',\n",
    "            n_jobs=-1,\n",
    "            verbose=0\n",
    "        )\n",
    "        \n",
    "        grid_search.fit(Xp, y_series)\n",
    "        \n",
    "        # Store results\n",
    "        self.best_params_ = grid_search.best_params_\n",
    "        self.best_score_ = grid_search.best_score_\n",
    "        self.cv_results_ = grid_search.cv_results_\n",
    "        self.best_estimator_ = grid_search.best_estimator_\n",
    "        self.pipeline = grid_search.best_estimator_\n",
    "        \n",
    "        # Store feature names for feature importance\n",
    "        self.feature_names_out_ = list(Xp.columns)\n",
    "        self.n_features_out_ = len(self.feature_names_out_)\n",
    "        \n",
    "        return self\n",
    "\n",
    "    def predict(self, X):\n",
    "        \"\"\"Predict fetal health classes.\"\"\"\n",
    "        Xp = self._prepare_X(X)\n",
    "        return self.pipeline.predict(Xp)\n",
    "\n",
    "    def predict_proba(self, X):\n",
    "        \"\"\"Return prediction probabilities for each class.\"\"\"\n",
    "        Xp = self._prepare_X(X)\n",
    "        return self.pipeline.predict_proba(Xp)\n",
    "\n",
    "    def get_feature_importance(self):\n",
    "        \"\"\"Return normalized feature importance scores sorted in descending order.\"\"\"\n",
    "        if not hasattr(self, 'pipeline'):\n",
    "            raise ValueError(\"Model must be fitted before getting feature importance\")\n",
    "        \n",
    "        # Get feature importance from the model\n",
    "        model = self.pipeline.named_steps['model']\n",
    "        importance = model.feature_importances_\n",
    "        \n",
    "        # Create Series with feature names\n",
    "        importance_series = pd.Series(\n",
    "            importance,\n",
    "            index=self.feature_names_out_\n",
    "        )\n",
    "        \n",
    "        # Normalize to sum to 1.0\n",
    "        importance_series = importance_series / importance_series.sum()\n",
    "        \n",
    "        # Sort in descending order\n",
    "        importance_series = importance_series.sort_values(ascending=False)\n",
    "        \n",
    "        return importance_series\n",
    "\n",
    "    def evaluate_per_class_metrics(self, X, y):\n",
    "        \"\"\"Return precision, recall, and F1-score for each class.\"\"\"\n",
    "        y_pred = self.predict(X)\n",
    "        y_true = pd.Series(y).squeeze()\n",
    "        \n",
    "        # Get unique classes\n",
    "        classes = sorted(np.unique(np.concatenate([y_true.unique(), y_pred])))\n",
    "        \n",
    "        # Calculate metrics per class\n",
    "        precision = precision_score(y_true, y_pred, labels=classes, average=None, zero_division=0)\n",
    "        recall = recall_score(y_true, y_pred, labels=classes, average=None, zero_division=0)\n",
    "        f1 = f1_score(y_true, y_pred, labels=classes, average=None, zero_division=0)\n",
    "        \n",
    "        # Create dictionary with results\n",
    "        metrics = {\n",
    "            'precision': dict(zip(classes, precision)),\n",
    "            'recall': dict(zip(classes, recall)),\n",
    "            'f1': dict(zip(classes, f1))\n",
    "        }\n",
    "        \n",
    "        return metrics\n",
    "\n",
    "    def get_plot_counts(self, X):\n",
    "        \"\"\"Return the exact counts used for the bar plot based on predicted fetal_health.\"\"\"\n",
    "        y_pred = pd.Series(self.predict(X), name=\"fetal_health\")\n",
    "        return y_pred.value_counts().sort_index()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Example Usage\n",
    "\n",
    "Below are example cells showing how to use the classifier:"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Load Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load your fetal health data\n",
    "# df = pd.read_csv('fetal_health.csv')\n",
    "# X = df.drop('fetal_health', axis=1)\n",
    "# y = df['fetal_health']"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Train the Model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize and train the classifier\n",
    "# classifier = PrenatalRiskClassifier()\n",
    "# classifier.fit(X, y)\n",
    "# print(f\"Best CV Score: {classifier.best_score_:.4f}\")\n",
    "# print(f\"Best Parameters: {classifier.best_params_}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Make Predictions"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Make predictions\n",
    "# predictions = classifier.predict(X)\n",
    "# print(f\"Predictions shape: {predictions.shape}\")\n",
    "# print(f\"Unique classes predicted: {np.unique(predictions)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Get Prediction Probabilities"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get prediction probabilities\n",
    "# probabilities = classifier.predict_proba(X)\n",
    "# print(f\"Probabilities shape: {probabilities.shape}\")\n",
    "# print(f\"First 5 prediction probabilities:\\n{probabilities[:5]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Analyze Feature Importance"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get feature importance\n",
    "# feature_importance = classifier.get_feature_importance()\n",
    "# print(\"Top 10 Most Important Features:\")\n",
    "# print(feature_importance.head(10))\n",
    "\n",
    "# # Visualize feature importance\n",
    "# import matplotlib.pyplot as plt\n",
    "# plt.figure(figsize=(10, 8))\n",
    "# feature_importance.head(15).plot(kind='barh')\n",
    "# plt.xlabel('Importance Score')\n",
    "# plt.title('Top 15 Feature Importances')\n",
    "# plt.tight_layout()\n",
    "# plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Evaluate Per-Class Metrics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluate per-class metrics\n",
    "# metrics = classifier.evaluate_per_class_metrics(X, y)\n",
    "# print(\"\\nPer-Class Metrics:\")\n",
    "# for metric_name, class_scores in metrics.items():\n",
    "#     print(f\"\\n{metric_name.upper()}:\")\n",
    "#     for class_label, score in class_scores.items():\n",
    "#         print(f\"  Class {class_label}: {score:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Get Prediction Distribution"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get prediction counts for visualization\n",
    "# counts = classifier.get_plot_counts(X)\n",
    "# print(\"\\nPrediction Distribution:\")\n",
    "# print(counts)\n",
    "\n",
    "# # Visualize prediction distribution\n",
    "# import matplotlib.pyplot as plt\n",
    "# plt.figure(figsize=(8, 6))\n",
    "# counts.plot(kind='bar')\n",
    "# plt.xlabel('Fetal Health Class')\n",
    "# plt.ylabel('Count')\n",
    "# plt.title('Distribution of Predicted Fetal Health Classes')\n",
    "# plt.xticks(rotation=0)\n",
    "# plt.tight_layout()\n",
    "# plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Model Summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Print comprehensive model summary\n",
    "# print(\"=\"*60)\n",
    "# print(\"MODEL SUMMARY\")\n",
    "# print(\"=\"*60)\n",
    "# print(f\"Number of features (original): {len(classifier.features)}\")\n",
    "# print(f\"Number of features (with engineering): {classifier.n_features_out_}\")\n",
    "# print(f\"Best cross-validation score: {classifier.best_score_:.4f}\")\n",
    "# print(f\"\\nBest hyperparameters:\")\n",
    "# for param, value in classifier.best_params_.items():\n",
    "#     print(f\"  {param}: {value}\")\n",
    "# print(\"=\"*60)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}